# 🎬 anime_upscale на Google Colab — интерполяция RIFE + x265

**Что делает ноутбук:** предпусковая проверка среды (GPU / интернет / g++ / ffmpeg / диск — fail-fast
с русским вердиктом), **обязательный проб драйвера** (на Colab драйвер 550-класса, CUDA 12.x →
автоматический swap torch на индекс cu126 с пинами `<2.15`), клон репозитория в `/content/anime_upscale`,
uv + зависимости, лестница деградации TensorRT (`trt=false`, если torch-tensorrt недоступен),
misc-плагин (SCDetect), модели, запуск и копирование результатов в Google Drive.

**Пререквизиты:** аккаунт Google; `Runtime → Change runtime type → T4 GPU`. Подробности — [notebooks/README.md](README.md).

**Идемпотентность:** ячейку 1 можно перезапускать — существующий клон переиспользуется (resume).
Диск `/content` эфемерный: `.venv` и результаты живут одну сессию — персистентность только через Drive.


In [ ]:
# ruff: noqa: T201, D103, E501, PLW1510, PLR2004, S607, S603, S310, FBT001, EM102, TRY003, TRY300, FBT002
# (ноутбуки вне src-гейта: интерактивный RU-баннер = легитимный print; список кодов = гистограмма прогона)
# Ячейка 1 — предпусковая проверка среды (fail-fast) + клон + uv + misc-плагин + модели.
# Идемпотентна: повторный запуск безопасен (клон переиспользуется, uv sync докатывает недостающее).
# ANIME_NOTEBOOK_DRY=1 — сухой прогон без сети и мутаций (локальная проверка самой ячейки).

import os
import re
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

# ── Константы (при необходимости правьте под себя) ─────────────────────────
REPO_URL = (
    "https://github.com/4q4r/anime_upscale.git"  # публичное репо; при закрытии доступа — свой форк
)
SEED_TRT_OFF = False  # True = принудительно trt=false, если сборка TRT-движка падает на прогоне
DRY = os.environ.get("ANIME_NOTEBOOK_DRY") == "1"

WORK_ROOT = Path(
    os.environ.get("ANIME_NOTEBOOK_WORKROOT", "/content")
)  # эфемерный каталог Colab (пропадает с сессией; персистентность — Drive)
CLONE_DIR = WORK_ROOT / "anime_upscale"
PLATFORM = "Google Colab"
SESSION_LIMIT = "~5 ч free tier, без гарантий — сессия может оборваться без предупреждения"
QUOTA_NOTE = "диск /content эфемерный, ~70–110 ГБ"
CU126_NOTE = "на Colab драйвер 550-класса (CUDA 12.x) → обязателен swap на cu126 + пины <2.15 (cu130-сборка torch на этом драйвере не запустится)"
GPU_FIX = "Runtime → Change runtime type → Hardware accelerator → T4 GPU"
NET_FIX = "Colab: интернет включен по умолчанию — проверьте соединение сессии"
DISK_MIN_GB = 3.0

CHECK, CROSS, WARN = "\u2713", "\u2717", "\u26a0"
failures: list[str] = []


def fail(msg: str) -> None:
    failures.append(msg)


def probe(label: str, ok: bool, detail_bad: str, remediation: str) -> bool:
    if ok:
        print(f"{CHECK} {label}")
    elif DRY:
        print(f"{WARN} {label}: {detail_bad} [DRY — не считается ошибкой]; фикс: {remediation}")
    else:
        print(f"{CROSS} {label}: {detail_bad}; фикс: {remediation}")
        fail(f"{label}: {detail_bad}")
    return ok


def run(
    cmd: list[str],
    cwd: Path | None = None,
    timeout: float | None = None,
    check: bool = True,
    env: dict[str, str] | None = None,
) -> int:
    print("$ " + " ".join(cmd) + (f"   (cwd={cwd})" if cwd else ""))
    if DRY:
        print("[DRY] выполнение пропущено")
        return 0
    proc = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, timeout=timeout, env=env)
    for line in (proc.stdout + proc.stderr).strip().splitlines()[-10:]:
        print(f"   {line}")
    if proc.returncode != 0:
        msg = f"команда завершилась с кодом {proc.returncode}: {' '.join(cmd)}"
        if check:
            raise SystemExit(f"{CROSS} {msg}")
        print(f"{WARN} {msg}")
    return proc.returncode


def gpu_facts() -> dict[str, str] | None:
    """nvidia-smi -> имя/драйвер/VRAM/compute-cap + CUDA-версия из шапки вывода."""
    if shutil.which("nvidia-smi") is None:
        return None
    try:
        csv = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name,driver_version,memory.total,compute_cap",
                "--format=csv,noheader",
            ],
            capture_output=True,
            text=True,
            timeout=30,
        )
        head = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
        if csv.returncode != 0 or head.returncode != 0 or not csv.stdout.strip():
            return None
        parts = [p.strip() for p in csv.stdout.splitlines()[0].split(",")]
        cuda = re.search(r"CUDA(?: UMD)? Version:\s*(\d+\.\d+)", head.stdout)
        return {
            "name": parts[0] if parts else "?",
            "driver": parts[1] if len(parts) > 1 else "?",
            "vram_mib": parts[2] if len(parts) > 2 else "?",
            "cc": parts[3] if len(parts) > 3 else "н/д",
            "cuda": cuda.group(1) if cuda else "?",
        }
    except (OSError, subprocess.SubprocessError):
        return None


def repo_reachable() -> bool:
    """Анонимная HEAD-проба репозитория: публичный → True, приватный/404 → False."""
    try:
        req = urllib.request.Request(REPO_URL.removesuffix(".git"), method="HEAD")
        urllib.request.urlopen(req, timeout=15)
        return True
    except OSError:
        return False


def free_gb(root: Path) -> float:
    probe_root = root if root.is_dir() else Path.cwd()
    return shutil.disk_usage(probe_root).free / 2**30


def yn(flag: object) -> str:
    return CHECK if flag else CROSS


# ── Пробы (сначала все, потом один честный вердикт) ────────────────────────
gpu = gpu_facts()
cuda_major = int(gpu["cuda"].split(".")[0]) if gpu and gpu["cuda"] != "?" else None

net_ok = False
try:
    urllib.request.urlopen(urllib.request.Request("https://github.com", method="HEAD"), timeout=15)
    net_ok = True
except OSError:
    net_ok = False

git_path, gxx_path, curl_path, ff_path = (shutil.which(t) for t in ("git", "g++", "curl", "ffmpeg"))
x265_ok = False
if ff_path:
    enc = subprocess.run(
        [ff_path, "-hide_banner", "-encoders"], capture_output=True, text=True, timeout=60
    )
    x265_ok = "libx265" in enc.stdout
disk_gb = free_gb(WORK_ROOT)

print()
print("=" * 20 + f" anime_upscale · {PLATFORM} · предпусковая проверка " + "=" * 20)
if gpu:
    print(f"GPU:          {gpu['name']}, {gpu['vram_mib']}, compute capability {gpu['cc']}")
    print(f"Драйвер:      {gpu['driver']}, CUDA {gpu['cuda']}")
else:
    print("GPU:          не обнаружен")
print(f"Python:       {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
print(f"git/g++/curl: {yn(git_path)} / {yn(gxx_path)} / {yn(curl_path)}")
print(f"ffmpeg:       {yn(ff_path)}, кодер libx265: {yn(x265_ok)}")
print(f"Диск:         свободно {disk_gb:.1f} ГБ ({QUOTA_NOTE})")
print(f"Лимит сессии: {SESSION_LIMIT}")
print(
    "Профиль:      interp-only — интерполяция RIFE (60 fps) → кодирование x265 10-bit, preset medium"
)
print()
print("БЮДЖЕТ (честная математика): x265 medium на 2–4 vCPU ≈ 1.5–4 fps при 1080p;")
print(
    "серия ~24 мин → 86 400 кадров при 60 fps → кодирование ≈ 6–16 ч (86 400 / 4 … 86 400 / 1.5 fps)."
)
print("Планируйте ПО ОДНОЙ серии на сессию: обрыв сессии теряет недокодированный файл.")
print("Хранилище:  /content эфемерный — результаты копируйте в Drive (последняя ячейка)")
print()
print(f"torch-ветка:  {CU126_NOTE}")
print()

probe("GPU", gpu is not None, "не обнаружен (nvidia-smi недоступен)", GPU_FIX)
probe("Интернет", net_ok, "нет доступа к github.com", NET_FIX)
probe(
    "git",
    git_path is not None,
    "не найден в PATH",
    "git предустановлен в образе сессии — перезапустите её",
)
probe("g++", gxx_path is not None, "не найден в PATH", "нужен для сборки misc-плагина (SCDetect)")
probe("curl", curl_path is not None, "не найден в PATH", "нужен build-скрипту misc-плагина")
probe(
    "ffmpeg",
    ff_path is not None,
    "не найден в PATH",
    "пайплайн кодирует через ffmpeg — в образах Kaggle/Colab он предустановлен",
)
if ff_path:
    probe(
        "libx265",
        x265_ok,
        "кодер libx265 отсутствует в этой сборке ffmpeg",
        "кодирование невозможно — возьмите сессию с полной apt-сборкой ffmpeg",
    )
probe(
    "Диск",
    disk_gb >= DISK_MIN_GB,
    f"свободно {disk_gb:.1f} ГБ (меньше {DISK_MIN_GB} ГБ)",
    "очистите прошлые результаты в рабочем каталоге или начните новую сессию",
)

if failures:
    print()
    print(f"{CROSS} ПРЕДПУСКОВАЯ ПРОВЕРКА ПРОВАЛЕНА — исправьте условия и перезапустите ячейку:")
    for item in failures:
        print(f"  {CROSS} {item}")
    raise SystemExit(1)

# ── Шаг 1: клон репозитория (существующий клон = resume) ───────────────────
print()
git_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")  # git не имеет права задавать вопросы
if DRY:
    print(f"[DRY] пропуск клонирования/обновления ({CLONE_DIR})")
elif (CLONE_DIR / ".git").is_dir():
    print("── Клон найден — переиспользую (resume), обновляю до origin/HEAD")
    run(["git", "fetch", "origin"], cwd=CLONE_DIR, timeout=300, env=git_env)
    run(["git", "reset", "--hard", "origin/HEAD"], cwd=CLONE_DIR, env=git_env)
    print(f"{CHECK} Клон обновлён: {CLONE_DIR}")
    print(f"   {WARN} локальные правки кода сброшены; config.toml правьте ПОСЛЕ этой ячейки")
else:
    print("── Клонирование репозитория")
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    clone_rc = run(
        ["git", "clone", REPO_URL, str(CLONE_DIR)],
        cwd=WORK_ROOT,
        timeout=600,
        env=git_env,
        check=False,
    )
    if clone_rc != 0:
        print(f"{CROSS} git clone завершился с кодом {clone_rc} — диагноз:")
        if not net_ok:
            print(
                f"  {CROSS} нет сети (проба github.com выше провалилась) — включите интернет в сессии"
            )
        elif shutil.which("git") is None:
            print(f"  {CROSS} git не найден в PATH")
        elif not repo_reachable():
            print(
                f"  {CROSS} {REPO_URL} недоступен анонимно (приватный/переименован?) — "
                "подставьте свой публичный форк в REPO_URL (константа ячейки 1)"
            )
        else:
            print(f"  {CROSS} сеть и репозиторий доступны — смотрите вывод git выше (диск/прокси)")
        raise SystemExit(1)
    print(f"{CHECK} Репозиторий склонирован: {CLONE_DIR}")
# ── Шаг 2: uv ──────────────────────────────────────────────────────────────
print("── Установка uv")
uv_path = shutil.which("uv")
if uv_path is None:
    if DRY:
        print("[DRY] пропуск установки uv")
    else:
        rc = run(
            [sys.executable, "-m", "pip", "install", "--quiet", "uv"], timeout=300, check=False
        )
        if rc != 0:
            print(f"{WARN} pip недоступен — официальный standalone-установщик astral.sh")
            run(["bash", "-c", "curl -LsSf https://astral.sh/uv/install.sh | sh"], timeout=300)
        uv_path = shutil.which("uv")
        if uv_path is None:
            for cand in (
                Path(sys.prefix) / "bin" / "uv",
                Path.home() / ".local" / "bin" / "uv",
                Path("/usr/local/bin") / "uv",
                Path.home() / ".cargo" / "bin" / "uv",
            ):
                if cand.is_file():
                    uv_path = str(cand)
                    os.environ["PATH"] = f"{cand.parent}{os.pathsep}{os.environ.get('PATH', '')}"
                    break
if uv_path is None:
    raise SystemExit(f"{CROSS} uv не найден после установки — добавьте каталог скриптов pip в PATH")
print(f"{CHECK} uv: {uv_path}")

# ── Шаг 3: torch-индекс под драйвер (cu126 + пины <2.15 при CUDA < 13) ─────
print("── Проверка torch-индекса")
pyproject = CLONE_DIR / "pyproject.toml"
if cuda_major is None:
    print(f"{WARN} CUDA-версия неизвестна (GPU не опознан) — pyproject клона не меняется")
elif cuda_major < 13:
    if DRY:
        print("[DRY] пропуск правки pyproject.toml (swap cu126 + пины <2.15)")
    else:
        text = pyproject.read_text(encoding="utf-8")
        patched = text.replace("whl/cu130", "whl/cu126")
        patched = patched.replace("CUDA-13.0 index", "CUDA-12.6 index")  # устаревший комментарий
        patched = re.sub(r'^([ \t]*)"torch",$', r'\1"torch<2.15",', patched, flags=re.MULTILINE)
        patched = re.sub(
            r'^([ \t]*)"torch-tensorrt",$', r'\1"torch-tensorrt<2.15",', patched, flags=re.MULTILINE
        )
        if patched != text:
            pyproject.write_text(patched, encoding="utf-8")
            print(
                f"{CHECK} CUDA {cuda_major}.x → индекс cu126 + пины <2.15 (эфемерно, только в клоне)"
            )
        else:
            print(f"{WARN} замены не применились — pyproject.toml клона отличается от ожидаемого")
else:
    print(f"{CHECK} CUDA {gpu['cuda']} → torch cu130 из lock-файла (без изменений)")

# ── Шаг 4: зависимости ─────────────────────────────────────────────────────
print("── uv sync (первый запуск скачивает ~3–6 ГБ torch)")
run([uv_path, "sync"], cwd=CLONE_DIR, timeout=3600)
print(f"{CHECK} Зависимости синхронизированы (.venv в клоне)")

# ── Шаг 5: TensorRT-проба и лестница деградации ────────────────────────────
config_path = CLONE_DIR / "config.toml"

if DRY:
    print("[DRY] пропуск TensorRT-пробы и деградации")
else:

    def seed_trt_off(reason: str) -> None:
        text = config_path.read_text(encoding="utf-8")
        if "trt = false" in text:
            print(f"{CHECK} trt = false уже стоит — деградация применена ранее, повтор не нужен")
            return
        if "trt = true" not in text:
            print(
                f"{WARN} в config.toml клона нет ключа trt — проверьте секцию [stages.interpolate] вручную"
            )
            return
        config_path.write_text(text.replace("trt = true", "trt = false", 1), encoding="utf-8")
        print(f"{CHECK} {reason} → [stages.interpolate] trt = false (PyTorch CUDA FP16; эфемерно)")

    if SEED_TRT_OFF:
        seed_trt_off("SEED_TRT_OFF=True (ручной флаг деградации)")
    else:
        print("── Проба torch-tensorrt")
        trt_rc = run(
            [uv_path, "run", "python", "-c", "import torch_tensorrt"],
            cwd=CLONE_DIR,
            timeout=900,
            check=False,
        )
        if trt_rc == 0:
            print(f"{CHECK} torch-tensorrt импортируется — интерполяция пойдёт через TensorRT")
            print(
                f"   {WARN} если прогон упадёт на сборке TRT-движка — выставьте SEED_TRT_OFF=True"
            )
        else:

            def drop_torch_tensorrt() -> None:
                text = pyproject.read_text(encoding="utf-8")
                text = re.sub(
                    r'^[ \t]*"torch-tensorrt[^"]*",[ \t]*\n', "", text, flags=re.MULTILINE
                )
                text = re.sub(
                    r'^torch-tensorrt = \{ index = "[^"]+" \}[ \t]*\n', "", text, flags=re.MULTILINE
                )
                pyproject.write_text(text, encoding="utf-8")

            drop_torch_tensorrt()
            run([uv_path, "sync"], cwd=CLONE_DIR, timeout=3600)
            seed_trt_off("torch-tensorrt не разрешился для этого Python/CUDA")

# ── Шаг 6: misc-плагин (SCDetect) — пересобирается после каждого uv sync ───
print("── Сборка misc-плагина (SCDetect)")
run(["bash", "scripts/build_misc_plugin.sh"], cwd=CLONE_DIR, timeout=900)
print(f"{CHECK} misc-плагин собран (core.misc доступен)")

# ── Шаг 7: модели из манифеста (sha256-пины) ───────────────────────────────
print("── Скачивание моделей по манифесту")
run([uv_path, "run", "anime-pipeline", "models", "fetch"], cwd=CLONE_DIR, timeout=1800)
print(f"{CHECK} Модели на месте (upscale + RIFE-сид)")

if not DRY:
    os.chdir(CLONE_DIR)
print()
print(f"{CHECK} Ячейка 1 готова — дальше ячейка запуска.")

# 📁 Ячейка 2 (опционально) — Google Drive

Монтируйте Drive, если нужен вход/выход/конфиг, пережиющий сессию. При `MOUNT_DRIVE = True`
Google покажет окно авторизации (штатный механизм Colab). Без Drive всё живёт в `/content`
и пропадает с сессией.


In [ ]:
# ruff: noqa: T201
# (ноутбуки вне src-гейта; коды = гистограмма прогона)
# Опционально: монтирование Google Drive (появится окно авторизации Google).
MOUNT_DRIVE = False  # True = смонтировать для персистентности input/config/output

if MOUNT_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    print("✓ Drive смонтирован: /content/drive/MyDrive")
    print("  Вход: положите файлы в MyDrive/anime_upscale_input и симлинком добавьте в ./input/")
else:
    print("Drive не монтирован (MOUNT_DRIVE = False) — всё живёт в /content и пропадёт с сессией.")

# ▶️ Ячейка 3 — запуск пайплайна

Замените `xxxx` на реальный код ссылки gofile. Варианты `-i`: ссылка gofile (папки разворачиваются
в батч), файл из `input/` (можно симлинком из Drive), несколько `-i` = батч. `--profile interp-only`
= интерполяция до 60 fps + x265 10-bit medium. Без `--upload` результат останется только локально.


In [ ]:
# Замените xxxx на код вашей ссылки gofile (или -i имя_файла.mkv после симлинка из Drive).
!cd "${ANIME_NOTEBOOK_WORKROOT:-/content}/anime_upscale" && uv run anime-pipeline --profile interp-only --upload -i https://gofile.io/d/xxxx

# 📦 Ячейка 4 — результаты

Листинг `output/`, напоминание про gofile-ссылки и честная оценка бюджета сессии по факту прогона.


In [ ]:
# ruff: noqa: T201, E501
# (ноутбуки вне src-гейта; коды = гистограмма прогона)
# Ячейка 4 — результаты: listing output/, ссылки gofile, честный бюджет сессии.
import json
import os
import subprocess
from pathlib import Path

CLONE_DIR = Path(os.environ.get("ANIME_NOTEBOOK_WORKROOT", "/content")) / "anime_upscale"
OUT_DIR = CLONE_DIR / "output"

files = sorted(OUT_DIR.glob("*.mkv")) if OUT_DIR.is_dir() else []
if not files:
    print("output/ пуст — ячейка запуска не выполнялась или упала (см. её вывод).")
for path in files:
    print(f"✓ {path.name} — {path.stat().st_size / 2**30:.2f} ГиБ")

print()
print(
    "Ссылки gofile (--upload) печатаются в выводе ячейки запуска — строки «https://gofile.io/d/…»."
)
print("Файлы в /content пропадут с сессией — скопируйте их в Drive (следующая ячейка).")

for path in files:
    try:
        proc = subprocess.run(
            ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", str(path)],
            capture_output=True,
            text=True,
            timeout=60,
        )
        duration_s = float(json.loads(proc.stdout)["format"]["duration"])
        frames_60 = round(duration_s * 60)
        lo_h = frames_60 / 4 / 3600  # 4 fps — optimistic 4-vCPU medium
        hi_h = frames_60 / 1.5 / 3600  # 1.5 fps — pessimistic 2-vCPU medium
        print(
            f"{path.name}: результат {duration_s / 60:.1f} мин; кадров при 60 fps ≈ {frames_60}; "
            f"серия такого объёма на 2–4 vCPU (medium, 1.5–4 fps) ≈ {lo_h:.1f}–{hi_h:.1f} ч кодирования."
        )
    except (OSError, subprocess.SubprocessError, KeyError, ValueError) as exc:
        print(f"{path.name}: ffprobe-оценка недоступна ({exc})")

print()
print("Напоминание: обрыв сессии теряет недокодированный файл — по одной серии за сессию.")

# ☁️ Ячейка 5 — копирование результатов в Drive

Копирует `output/*.mkv` в `MyDrive/anime_upscale_output/`. Требует смонтированный Drive
(ячейка 2 с `MOUNT_DRIVE = True`); без него — скачайте файлы вручную из файловой панели Colab.


In [ ]:
# ruff: noqa: T201
# (ноутбуки вне src-гейта; коды = гистограмма прогона)
# Копирование output/*.mkv в Google Drive (если Drive смонтирован).
import os
import shutil
from pathlib import Path

OUT_DIR = Path(os.environ.get("ANIME_NOTEBOOK_WORKROOT", "/content")) / "anime_upscale" / "output"
DRIVE_OUT = Path("/content/drive/MyDrive/anime_upscale_output")

if not Path("/content/drive/MyDrive").is_dir():
    print(
        "Drive не смонтирован — включите MOUNT_DRIVE = True в ячейке монтирования и выполните её,"
    )
    print("либо скачайте файлы вручную через файловую панель Colab (слева).")
else:
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    files = sorted(OUT_DIR.glob("*.mkv")) if OUT_DIR.is_dir() else []
    if not files:
        print("output/ пуст — копировать нечего.")
    for path in files:
        dst = DRIVE_OUT / path.name
        shutil.copy2(path, dst)
        print(f"✓ {path.name} → {dst} ({path.stat().st_size / 2**20:.1f} МиБ)")